# Task 6 — House Price Prediction (California Housing)

## Objective
Train a regression model to predict California housing prices using the `fetch_california_housing()` dataset.

## Dataset
- Source: `sklearn.datasets.fetch_california_housing()`
- Features: demographic + geographic variables
- Target: median house value (in 100,000s)

## Approach
1. Load dataset into a DataFrame
2. EDA: histograms, correlation heatmap, pairplot (sample if > 5000 rows)
3. Preprocess: `StandardScaler`
4. Train/test split: 80/20 (random_state=42)
5. Model: Gradient Boosting Regressor (fallback: Random Forest)
6. Hyperparameter tuning: `GridSearchCV` over `n_estimators`, `max_depth`
7. Evaluate: MAE, RMSE, R²
8. Visualize: actual vs predicted + feature importances

## Final Summary
This notebook provides a complete baseline for tabular regression with EDA, scaling, tuning, and evaluation plots.

In [ ]:
# pip install pandas numpy scikit-learn matplotlib seaborn

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
df.head()

In [ ]:
print('Shape:', df.shape)
df.describe().T.head(10)

## EDA
- Histograms: distribution of each feature
- Correlation heatmap
- Pairplot: sampled if dataset is large (more than 5000 rows)

In [ ]:
df.hist(figsize=(12, 10), bins=30)
plt.suptitle('Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
sample_df = df.sample(n=min(len(df), 5000), random_state=42) if len(df) > 5000 else df
sns.pairplot(sample_df, corner=True, diag_kind='hist')
plt.show()

In [ ]:
target_name = 'MedHouseVal'
X = df.drop(columns=[target_name])
y = df[target_name]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape

In [ ]:
def build_pipeline(model):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('model', model),
    ])

# Prefer Gradient Boosting; fallback to RandomForest if something fails
try:
    base_model = GradientBoostingRegressor(random_state=42)
    pipe = build_pipeline(base_model)
    param_grid = {
        'model__n_estimators': [100, 300],
        'model__max_depth': [2, 3],
    }
except Exception as e:
    print('GradientBoosting setup failed; falling back to RandomForest. Error:', e)
    base_model = RandomForestRegressor(random_state=42, n_jobs=-1)
    pipe = build_pipeline(base_model)
    param_grid = {
        'model__n_estimators': [200, 500],
        'model__max_depth': [None, 10],
    }

search = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=3,
    n_jobs=-1,
)
search.fit(X_train, y_train)

print('Best params:', search.best_params_)
best_model = search.best_estimator_
best_model

In [ ]:
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print(f'MAE:  {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R^2:  {r2:.4f}')

In [ ]:
# Plot: Actual vs Predicted
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.4)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', linewidth=2)
plt.xlim(lims)
plt.ylim(lims)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted (Identity Line)')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (if supported)
model_step = best_model.named_steps['model']
if hasattr(model_step, 'feature_importances_'):
    importances = model_step.feature_importances_
    fi = pd.Series(importances, index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=fi.values, y=fi.index)
    plt.title('Feature Importances')
    plt.tight_layout()
    plt.show()
else:
    print('Model does not expose feature_importances_.')